# Student + GradingSystem: Demonstrating Python Dunder Methods

This notebook demonstrates how Python's "dunder" (double-underscore) methods allow classes to behave like built-in types.

We use a simple domain:
- **Student**: a name and 10 quiz scores
- **GradingSystem**: a collection of students

You will see how classes can implement methods like:
- `__str__`, `__repr__`
- `__len__`, `__iter__`, `__contains__`
- `__getitem__`
- `__eq__`, `__hash__`, `__lt__`
- `__bool__`, `__reversed__`, `__add__`

These methods make your objects feel *native* in Python, so they work naturally with `len()`, `sorted()`, `in`, indexing, and more.

## ColorTheme

We start with a small helper class, `ColorTheme`, which provides ANSI color codes for terminal output.

This is **not** essential to the logic of the grading system. Its main purpose is to make it visually obvious when different dunder methods are being called, by printing colored traces.

You can ignore the color details if you want to focus purely on the dunder behavior.

In [ ]:
# ColorTheme: provides ANSI color codes for printing

class ColorTheme:
    def __init__(self, custom=None):
        ORANGE = "\033[38;5;208m"
        RESET = "\033[0m"
        print(f"{ORANGE}[{self.__class__.__name__}] → __init__{RESET}")

        base = {
            "blue": "\033[94m",
            "green": "\033[92m",
            "yellow": "\033[93m",
            "cyan": "\033[96m",
            "red": "\033[91m",
            "magenta": "\033[95m",
            "orange": "\033[38;5;208m",
            "lavender": "\033[38;5;183m\033[1m",
            "grey": "\033[38;5;245m",
            "bold": "\033[1m",
            "white": "\033[1;97m",
            "reset": "\033[0m",
        }
        self._theme = base if custom is None else {**base, **custom}

    def get(self, name):
        return self._theme.get(name.lower(), self._theme["reset"])


# Global theme instance used by other classes
c = ColorTheme()

## Student Class

The `Student` class models:
- a student name
- exactly 10 quiz scores
- methods to compute average, highest, and lowest scores

More importantly for this notebook, `Student` implements several dunder methods so that:

- `__str__` and `__repr__` control how the object is printed
- `__len__` returns the number of scores
- `__iter__` lets you loop over scores directly
- `__getitem__` lets you index into scores (e.g., `student[0]`)
- `__contains__` lets you check if a score is present (e.g., `100 in student`)
- `__eq__` and `__hash__` treat students with the same name as equal
- `__lt__` allows students to be ordered by average score

Each dunder prints a colored trace when called, so you can see exactly when Python is using them.

In [ ]:
# Student: models a student and their quiz scores, with rich dunder behavior

class Student:
    def __init__(self, name, scores):
        print(f"{c.get('orange')}[Student] → __init__{c.get('reset')}")
        self._validate_name(name)
        self._validate_scores(scores)
        self._name = name
        self._scores = scores.copy()

    # Validation helpers
    def _validate_name(self, name):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Student name must be a non-empty string.")

    def _validate_scores(self, scores):
        if not isinstance(scores, list):
            raise TypeError("Scores must be a list.")
        if len(scores) != 10:
            raise ValueError("Must have exactly 10 quiz scores.")
        for i, s in enumerate(scores):
            if not isinstance(s, (int, float)) or not (0 <= s <= 100):
                raise ValueError(f"Score {i} must be between 0 and 100.")

    # Public API
    def get_name(self):
        return self._name

    def get_scores(self):
        return self._scores.copy()

    def average(self):
        return sum(self._scores) / len(self._scores)

    def highest(self):
        return max(self._scores)

    def lowest(self):
        return min(self._scores)

    # Dunder methods
    def __str__(self):
        print(f"{c.get('orange')}[Student] → __str__{c.get('reset')}")
        return f"{self._name} (avg={self.average():.1f})"

    def __repr__(self):
        print(f"{c.get('orange')}[Student] → __repr__{c.get('reset')}")
        return f"Student(name={self._name!r}, scores={self._scores})"

    def __eq__(self, other):
        print(f"{c.get('orange')}[Student] → __eq__{c.get('reset')}")
        if not isinstance(other, Student):
            return NotImplemented
        return self._name.lower() == other._name.lower()

    def __hash__(self):
        print(f"{c.get('orange')}[Student] → __hash__{c.get('reset')}")
        return hash(self._name.lower())

    def __lt__(self, other):
        print(f"{c.get('orange')}[Student] → __lt__{c.get('reset')}")
        if not isinstance(other, Student):
            return NotImplemented
        return self.average() < other.average()

    def __getitem__(self, index):
        print(f"{c.get('orange')}[Student] → __getitem__{c.get('reset')}")
        return self._scores[index]

    def __len__(self):
        print(f"{c.get('orange')}[Student] → __len__{c.get('reset')}")
        return len(self._scores)

    def __iter__(self):
        print(f"{c.get('orange')}[Student] → __iter__{c.get('reset')}")
        return iter(self._scores)

    def __contains__(self, value):
        print(f"{c.get('orange')}[Student] → __contains__{c.get('reset')}")
        return value in self._scores

    def __getattr__(self, item):
        print(f"{c.get('orange')}[Student] → __getattr__({item}){c.get('reset')}")
        raise AttributeError(f"Student has no attribute '{item}'")

    def __dir__(self):
        print(f"{c.get('orange')}[Student] → __dir__{c.get('reset')}")
        return sorted(list(self.__dict__.keys()) + ["average", "highest", "lowest"])

## GradingSystem Class

The `GradingSystem` class manages a collection of `Student` objects.

It is designed to behave like a Python container:

- `__len__` → number of students
- `__iter__` → iterate over students
- `__contains__` → check if a student with a given name exists
- `__getitem__` → index by position or look up by name
- `__str__` and `__repr__` → human-readable and debug representations
- `__eq__` → compare two grading systems by their set of student names
- `__bool__` → `True` if there are any students, `False` if empty
- `__reversed__` → iterate over students in reverse order
- `__add__` → merge two grading systems into a new one

Again, each dunder prints a trace when called so you can see how Python uses them behind the scenes.

In [ ]:
# GradingSystem: manages a collection of students, with rich dunder behavior

class GradingSystem:
    def __init__(self):
        print(f"{c.get('orange')}[GradingSystem] → __init__{c.get('reset')}")
        self._students = []

    # API methods
    def add_students(self, data):
        for entry in data:
            self._students.append(Student(entry["name"], entry["scores"]))

    def find(self, name):
        for s in self._students:
            if s.get_name().lower() == name.lower():
                return s
        return None

    def class_average(self):
        if not self._students:
            return 0.0
        return sum(s.average() for s in self._students) / len(self._students)

    # Dunder methods
    def __len__(self):
        print(f"{c.get('orange')}[GradingSystem] → __len__{c.get('reset')}")
        return len(self._students)

    def __iter__(self):
        print(f"{c.get('orange')}[GradingSystem] → __iter__{c.get('reset')}")
        return iter(self._students)

    def __contains__(self, name):
        print(f"{c.get('orange')}[GradingSystem] → __contains__{c.get('reset')}")
        return any(s.get_name().lower() == name.lower() for s in self._students)

    def __getitem__(self, key):
        print(f"{c.get('orange')}[GradingSystem] → __getitem__{c.get('reset')}")
        if isinstance(key, (int, slice)):
            return self._students[key]
        if isinstance(key, str):
            return self.find(key)
        raise TypeError("Index must be int, slice, or str.")

    def __str__(self):
        print(f"{c.get('orange')}[GradingSystem] → __str__{c.get('reset')}")
        names = ", ".join(s.get_name() for s in self._students)
        return f"GradingSystem: {names}"

    def __repr__(self):
        print(f"{c.get('orange')}[GradingSystem] → __repr__{c.get('reset')}")
        return f"GradingSystem({len(self._students)} students)"

    def __eq__(self, other):
        print(f"{c.get('orange')}[GradingSystem] → __eq__{c.get('reset')}")
        if not isinstance(other, GradingSystem):
            return NotImplemented
        return set(s.get_name().lower() for s in self._students) == \
               set(s.get_name().lower() for s in other._students)

    def __bool__(self):
        print(f"{c.get('orange')}[GradingSystem] → __bool__{c.get('reset')}")
        return len(self._students) > 0

    def __reversed__(self):
        print(f"{c.get('orange')}[GradingSystem] → __reversed__{c.get('reset')}")
        return reversed(self._students)

    def __add__(self, other):
        print(f"{c.get('orange')}[GradingSystem] → __add__{c.get('reset')}")
        if not isinstance(other, GradingSystem):
            return NotImplemented
        new = GradingSystem()
        new._students = self._students + other._students
        return new

## Demonstration: Triggering Dunder Methods

The `main()` function below creates a `GradingSystem`, adds some sample students, and then exercises many of the dunder methods.

As you run it, watch the colored output to see when each dunder is called:

- `len(gs)` → `__len__`
- `for student in gs` → `__iter__`
- `'Alice' in gs` → `__contains__`
- `gs[0]` and `gs['Carol']` → `__getitem__`
- `sorted(gs)` → `Student.__lt__`
- `hash(student)` and `student1 == student2` → `__hash__`, `__eq__`
- `bool(gs)` → `__bool__`
- `reversed(gs)` → `__reversed__`
- `gs + gs2` → `__add__`
- `gs['Alice'][0]` → `Student.__getitem__`
- `for score in gs['Bob']` → `Student.__iter__`
- `100 in gs['Carol']` → `Student.__contains__`

In [ ]:
def main():
    print("\n" + c.get("bold") + "=== Demonstrating Dunders ===" + c.get("reset") + "\n")

    data = [
        {"name": "Alice", "scores": [90, 92, 88, 95, 100, 85, 87, 93, 91, 89]},
        {"name": "Bob",   "scores": [70, 75, 72, 68, 80, 77, 74, 73, 71, 69]},
        {"name": "Carol", "scores": [98, 97, 99, 96, 95, 94, 93, 92, 91, 90]},
    ]

    gs = GradingSystem()
    gs.add_students(data)

    print("\n--- __str__ ---")
    print(gs)

    print("\n--- __len__ ---")
    print(len(gs))

    print("\n--- __iter__ ---")
    for student in gs:
        print(student)

    print("\n--- __contains__ ---")
    print("Alice" in gs)
    print("Xavier" in gs)

    print("\n--- __getitem__ ---")
    print(gs[0])
    print(gs["Carol"])

    print("\n--- Student __lt__ (sorting) ---")
    sorted_students = sorted(gs)
    for s in sorted_students:
        print(s)

    print("\n--- Student __hash__ + __eq__ ---")
    s1 = gs["Alice"]
    s2 = Student("Alice", [100]*10)
    print(hash(s1), hash(s2))
    print(s1 == s2)

    print("\n--- __bool__ ---")
    print(bool(gs))

    print("\n--- __reversed__ ---")
    for s in reversed(gs):
        print(s)

    print("\n--- __add__ (merge two grading systems) ---")
    gs2 = GradingSystem()
    gs2.add_students([
        {"name": "Dave", "scores": [50]*10}
    ])
    merged = gs + gs2
    print(merged)

    print("\n--- Student score indexing (__getitem__) ---")
    print(gs["Alice"][0])  # first quiz score

    print("\n--- Student iteration (__iter__) ---")
    for score in gs["Bob"]:
        print(score)

    print("\n--- Student __contains__ ---")
    print(100 in gs["Carol"])
    print(42 in gs["Carol"])

    print("\n=== End Demonstration ===\n")

## Run the Demonstration

Now run the cell below to execute `main()` and watch the dunder methods being called.

You can re-run this cell multiple times to see the traces again.

In [ ]:
main()